# Day 2 notebook companion

Run in order with synthetic data. Mermaid diagrams render on the website. Setup and shared helpers are embedded; no checkout is required. Learner exercises report NOT ATTEMPTED until implemented. Reference checks are separate. Optional controls also have direct function calls.


In [ ]:
import importlib.metadata
import subprocess
import sys
for package, version in {"cryptography": "50.0.1", "matplotlib": "3.10.6", "ipywidgets": "8.1.7"}.items():
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        subprocess.check_call([sys.executable, "-m", "pip", "install", f"{package}=={version}"])
print("Dependencies ready. Restart if an older library was already imported, then run all cells.")


## Shared teaching helpers

Inspect this implementation. TLS uses real SSL objects over memory buffers and temporary test key files; no system trust changes or network listeners. The teaching KDF is not a standardized protocol key schedule.


In [ ]:
"""Day 2 teaching helpers. Real TLS over MemoryBIO; no sockets or trust-store changes."""
from datetime import datetime, timedelta, timezone
from pathlib import Path
import ssl
import tempfile
import hashlib
from cryptography import x509
from cryptography.x509.oid import NameOID, ExtendedKeyUsageOID
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives.kdf.hkdf import HKDF


def make_pki(expired=False):
    """Create an isolated root, intermediate, server, and client for this run."""
    now = datetime.now(timezone.utc)
    keys = {name: ec.generate_private_key(ec.SECP256R1())
            for name in ('root', 'intermediate', 'server', 'client')}
    names = {name: x509.Name([x509.NameAttribute(NameOID.COMMON_NAME, 'Workshop ' + name)])
             for name in keys}
    certs = {}
    for name, issuer, ca, path_length, eku in [
        ('root', 'root', True, 1, None),
        ('intermediate', 'root', True, 0, None),
        ('server', 'intermediate', False, None, ExtendedKeyUsageOID.SERVER_AUTH),
        ('client', 'intermediate', False, None, ExtendedKeyUsageOID.CLIENT_AUTH),
    ]:
        end = now - timedelta(days=1) if expired and name == 'server' else now + timedelta(days=7)
        builder = (x509.CertificateBuilder().subject_name(names[name]).issuer_name(names[issuer])
                   .public_key(keys[name].public_key()).serial_number(x509.random_serial_number())
                   .not_valid_before(now - timedelta(days=2)).not_valid_after(end)
                   .add_extension(x509.BasicConstraints(ca=ca, path_length=path_length), critical=True)
                   .add_extension(x509.KeyUsage(digital_signature=True, content_commitment=False,
                       key_encipherment=False, data_encipherment=False, key_agreement=False,
                       key_cert_sign=ca, crl_sign=ca, encipher_only=False, decipher_only=False), critical=True)
                   .add_extension(x509.SubjectKeyIdentifier.from_public_key(keys[name].public_key()), False)
                   .add_extension(x509.AuthorityKeyIdentifier.from_issuer_public_key(keys[issuer].public_key()), False))
        if eku:
            builder = builder.add_extension(x509.ExtendedKeyUsage([eku]), False)
            builder = builder.add_extension(x509.SubjectAlternativeName([
                x509.DNSName('invoice.test' if name == 'server' else 'client.test')]), False)
        certs[name] = builder.sign(keys[issuer], hashes.SHA256())
    return keys, certs


def tls_trial(hostname='invoice.test', trust_root=True, expired=False,
              include_intermediate=True, mtls=False, send_client=True,
              client_wrong_eku=False):
    """Handshake and exchange application bytes. Failures raise ssl.SSLError.

    Private PEM files are disposable teaching keys in a temporary directory.
    Does not implement online revocation, networking, or authorization policy.
    """
    keys, certs = make_pki(expired)
    pem = lambda c: c.public_bytes(serialization.Encoding.PEM)
    with tempfile.TemporaryDirectory(prefix='workshop-pki-') as directory:
        base = Path(directory)
        for name in ('server', 'client'):
            selected = 'server' if name == 'client' and client_wrong_eku else name
            chain = pem(certs[selected])
            if include_intermediate or name == 'client':
                chain += pem(certs['intermediate'])
            (base / (name + '.pem')).write_bytes(chain)
            (base / (name + '.key')).write_bytes(keys[selected].private_bytes(
                serialization.Encoding.PEM, serialization.PrivateFormat.PKCS8,
                serialization.NoEncryption()))
        server_context = ssl.SSLContext(ssl.PROTOCOL_TLS_SERVER)
        client_context = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
        for context in (server_context, client_context):
            context.minimum_version = context.maximum_version = ssl.TLSVersion.TLSv1_3
        server_context.load_cert_chain(str(base / 'server.pem'), str(base / 'server.key'))
        if trust_root:
            client_context.load_verify_locations(cadata=pem(certs['root']).decode())
        if mtls:
            server_context.verify_mode = ssl.CERT_REQUIRED
            server_context.load_verify_locations(cadata=pem(certs['root']).decode())
        if send_client:
            client_context.load_cert_chain(str(base / 'client.pem'), str(base / 'client.key'))
        ci, co, si, so = (ssl.MemoryBIO() for _ in range(4))
        client = client_context.wrap_bio(ci, co, server_hostname=hostname)
        server = server_context.wrap_bio(si, so, server_side=True)
        completed = [False, False]

        def transfer():
            for outgoing, incoming in ((co, si), (so, ci)):
                if outgoing.pending:
                    incoming.write(outgoing.read())

        for _ in range(100):
            for index, peer in enumerate((client, server)):
                if not completed[index]:
                    try:
                        peer.do_handshake()
                        completed[index] = True
                    except ssl.SSLWantReadError:
                        pass
            transfer()
            if all(completed):
                break
        else:
            raise RuntimeError('TLS handshake stalled')
        payload = b'synthetic confidential invoice'
        client.write(payload)
        transfer()
        assert server.read(4096) == payload
        return {'version': client.version(), 'cipher': client.cipher()[0],
                'client_authenticated': bool(server.getpeercert()),
                'application_bytes': len(payload)}


def expect_rejection(operation, exceptions):
    """Assert the negative case, without accepting a silent failure."""
    try:
        operation()
    except exceptions:
        return
    raise AssertionError('Expected rejection did not occur')


def derive_day2(secret, transcript, direction=b'alice-to-bob'):
    """Teaching KDF only, not a standardized TLS or hybrid key schedule."""
    return HKDF(algorithm=hashes.SHA256(), length=32, salt=None,
                info=b'workshop-day2:v1|' + hashlib.sha256(transcript).digest()
                + b'|' + direction).derive(secret)


# Session 11: Post-Quantum Digital Signatures

**45 minutes taught · 75–100 minutes independently.** Instructor (see course website) · Notebook (see course website)

## Outcomes and setup

Generate and verify ML-DSA signatures, reject altered messages/contexts/keys, measure public signature material, and explain the trust changes required for signed updates and certificates. Complete Session 5 and use the Day 2 setup (see course website). The examples use ML-DSA-65 from the pinned library.



## What changes and what stays

ML-DSA is a module-lattice signature family standardized in FIPS 204. It replaces a signature primitive; it does not encrypt data or establish a KEM secret. A valid signature remains relative to a trusted public key and exact signed content. Authorization, freshness, release policy and key custody still matter.

```mermaid
flowchart LR
    M["Message and specified context"] --> S["ML-DSA signing with private key"]
    S --> V["ML-DSA verification"]
    K["Authentic public verification key"] --> V
    M --> V
    V --> P["Application authorization and freshness policy"]
```

Read the final arrow carefully: post-quantum verification does not turn a wrong-product update into an acceptable release. It also cannot prove that an uncompromised human approved a message if the signing service was misused.


In [ ]:
from cryptography.hazmat.primitives.asymmetric.mldsa import MLDSA65PrivateKey
from cryptography.exceptions import InvalidSignature
signer = MLDSA65PrivateKey.generate()
public = signer.public_key()
message = b'release:v1|product=training-device|version=9|digest=synthetic'
context = b'workshop-release'
signature = signer.sign(message, context)
public.verify(signature, message, context)
assert (len(public.public_bytes_raw()), len(signature)) == (1952, 3309)
print('PASS: ML-DSA-65 signature verifies; public key/signature = 1952/3309 bytes')


This API takes an optional context; both sides must agree on it. Use the protocol's context, variant and encoding. Do not substitute a manually prehashed message or the library's lower-level message-representative API without a specification requiring that operation.


In [ ]:
other = MLDSA65PrivateKey.generate()
for operation in [
    lambda: public.verify(signature, message + b'!', context),
    lambda: public.verify(signature, message, b'other-purpose'),
    lambda: other.public_key().verify(signature, message, context),
    lambda: public.verify(signature[:-1], message, context),
]:
    expect_rejection(operation, InvalidSignature)
print('PASS: altered message, context, key and truncated signature are rejected')


## Measure the deployment impact


In [ ]:
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey
import matplotlib.pyplot as plt
ed = Ed25519PrivateKey.generate()
sizes = [len(ed.sign(message)), len(signature)]
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(['Ed25519', 'ML-DSA-65'], sizes)
ax.set(ylabel='Signature bytes', title='Measured raw signature lengths — not equal-security benchmarking')
fig.tight_layout()
plt.show()
assert sizes == [64, 3309]
print('PASS: compared serialized signature lengths without claiming equal security or runtime')


This graph does not compare throughput, verification time or equivalent security categories. Certificate chains contain more than one signature plus public keys, names and extensions. Measure full objects and paths through proxies, bootloaders, storage slots and update transports. A format's maximum size can be as important as its average transfer time.

```mermaid
xychart-beta
    title "Raw signature bytes in the teaching profiles"
    x-axis [Ed25519, ML-DSA-65]
    y-axis "Bytes" 0 --> 3500
    bar [64, 3309]
```

The website chart displays the same measured sizes as the notebook. The profiles are not presented as equal-security alternatives.

| Family | Main idea | Deployment question |
| --- | --- | --- |
| ML-DSA | Module-lattice signatures; 44, 65 and 87 parameter sets | Which parameter set, context, encoding and platform support are required? |
| SLH-DSA | Stateless hash-based signatures standardized in FIPS 205 | How do larger signatures and selected speed/size parameters fit the target? |
| Classical signatures | Existing RSA/ECC trust ecosystems | Which verifiers and trust anchors must change, and what remains vulnerable? |

SLH-DSA offers different underlying assumptions from lattice signatures. It is not interchangeable with ML-DSA at the API or object-format level. This course gives a conceptual SLH-DSA overview, not an executable implementation. Stateless does not mean keys need no lifecycle management.

## Migrate verification before relying on new signatures

```mermaid
flowchart TD
    I["Inventory verifiers and object limits"] --> T["Provision authentic PQ verification capability"]
    T --> P["Specify signature acceptance policy"]
    P --> R["Issue and distribute signed artifacts"]
    R --> M["Monitor verification and recovery paths"]
```

A firmware device cannot benefit from an ML-DSA-signed update if its immutable verifier only understands a classical scheme. Trust updates and boot-chain support must precede reliance on the new signature. A certificate ecosystem also needs compatible issuance, parsing, validation and handshake signature support; merely generating an ML-DSA key does not update TLS.

If a transition distributes both classical and PQ signatures, define whether verifiers require **both**, permit **either**, or apply a versioned policy. “Either” can retain acceptance through a broken component; “both” can break availability when a verifier lacks support. This is an explicit protocol and rollout decision, not an accidental `or` in code.

For the state actor scenario, protect build inputs, release approval and signing-service permissions. An attacker authorized to call a PQ signing key can produce valid malicious releases just as with classical keys. Future forgery risk also differs from HNDL decryption: archival authenticity may require trustworthy timing and preservation evidence, not just a new encryption key.

## Practice and answers

1. Will changing the context after signing preserve validity?
2. Does valid ML-DSA make a firmware version current or authorized?
3. What breaks if an update parser allocates only 512 bytes for a signature?
4. A dual-signature verifier accepts either signature. Does it preserve the intended security if the classical one becomes forgeable?

<details><summary>Worked answers</summary>
<ol><li>No. This signature binds its context.</li><li>No. Product, version, authorization and replay/rollback policy remain.</li><li>The ML-DSA-65 signature will not fit; test format limits and fail safely rather than truncate.</li><li>Not for acceptance that still permits a forged classical signature alone. Specify the transition policy and test removal of either signature.</li></ol>
</details>

Continue to Lab 6 (see course website) and then the Day 3 architecture material (see course website).

## Sources

Reviewed 22 September 2026: [FIPS 204 and errata notices](https://csrc.nist.gov/pubs/fips/204/final), [FIPS 205](https://csrc.nist.gov/pubs/fips/205/final), [ML-DSA API](https://cryptography.io/en/stable/hazmat/primitives/asymmetric/mldsa/). Standardization of a primitive is not a claim that this Python runtime is FIPS validated.


In [ ]:
print("PASS: completed session-11-ml-dsa demonstrations; learner status is reported separately")
